# Microsoft Agent Framework — Azure OpenAI (Responses API)

Tässä koodiesimerkissä käytät **Microsoft Agent Frameworkia (MAF)** luodaksesi yksinkertaisen agentin, jota tukee **Azure OpenAI** käyttäen **Responses API:a**.

> **Siirtymismuistutus:** Tämä esimerkki käytti aiemmin Semantic Kernelia GitHub-mallien kanssa. Se on siirretty Microsoft Agent Frameworkiin, ja GitHub-mallit (vanhentumassa, poistumassa heinäkuussa 2026) on korvattu Azure OpenAI:lla, joka tukee Responses API:a. MAF:n `OpenAIChatClient` käyttää oletuksena Azure OpenAI:n vakaata `/openai/v1/` -päätepistettä ja Responses API:a.

Tämän esimerkin tarkoituksena on näyttää vaiheet, jotka sovelletaan myöhemmin lisäesimerkeissä erilaisten agenttimalleiden toteuttamisessa.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Tuo tarvittavat Python-kirjastot


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Työkalun määrittely

Microsoft Agent Frameworkissa **työkalu** on tavallinen Python-funktio, joka on koristeltu `@tool`-merkinnällä ja jota agentti voi kutsua. Alla määrittelemme työkalun, joka palauttaa satunnaisen lomakohteen ja välttää aiemman kohteen toistamisen.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Agentin luominen

Tässä luomme agentin nimeltä `TravelAgent`.

Tässä esimerkissä käytämme hyvin yksinkertaisia ohjeita. Muokkaa näitä ohjeita vapaasti nähdäksesi, miten agentin käyttäytyminen muuttuu.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Agentin suorittaminen

Nyt voimme suorittaa agentin. Luomme `AgentSession`-olion, jotta agentti muistaa keskustelun vuorojen välillä, ja lähetämme sitten kaksi `user_inputs`-viestiä. Ensimmäinen pyytää matkaa; toinen kertoo, ettei käyttäjä pitänyt ehdotuksesta ja pyytää toista — agentti käyttää istuntokokemusta sekä `get_random_destination`-työkalua vastatessaan.

Voit muuttaa näitä viestejä nähdäksesi, miten agentti reagoi eri tavoin. Vastaukset **striimataan** token tokenilta.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
